Save a copy of this notebook, and then USE THAT COPY.
Put your images into a folder in your gdrive; you'll use the `cp` command to copy FROM your gdrive subfolder TO the colab notebook space.

In [1]:
# verify
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq libgl1 libgomp1 libxcb-cursor0

Sun Nov 22 20:52:45 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content
!curl -L -o meshroom.tar.gz "https://github.com/alicevision/Meshroom/releases/download/v2023.3.0/Meshroom-2023.3.0-linux.tar.gz"
!tar -xzf meshroom.tar.gz && rm meshroom.tar.gz
!mv Meshroom-2023.3.0 meshroom

/content
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 1187M  100 1187M    0     0  36.2M      0  0:00:32  0:00:32 --:--:-- 41.8M


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
!mkdir -p /content/images /content/output /content/cache
# Change the Drive path below to your folder of photos
!cp -r "/content/drive/MyDrive/my_photogrammetry_images/." /content/images/


# now we run meshroom in the background on your images

This might take awhile. You can try googline for the 2023 meshroom CLI to find out what other settings you can tweak (not the 2025 version).

In [10]:
%env QT_QPA_PLATFORM=offscreen
!/content/meshroom/meshroom_batch \
  -i /content/images \
  -o /content/output \
  --cache /content/cache \
  -p photogrammetry \
  --paramOverrides Texturing:colorMapping.colorMappingFileType=jpg \
  -v info

env: QT_QPA_PLATFORM=offscreen
[2026-11-22 21:04:41.890692] [0x00007ff7c8054000] [trace]   Embedded OCIO configuration file: '/content/meshroom/aliceVision/share/aliceVision/config.ocio' found.
Program called with the following parameters:
 * allowSingleView = 1
 * allowedCameraModels = "pinhole,radial1,radial3,brown,fisheye4,fisheye1,3deanamorphic4,3deradial4,3declassicld"
 * colorProfileDatabase = "" (default)
 * defaultCameraModel = "" (default)
 * defaultFieldOfView = 45
 * defaultFocalLength = -1 (default)
 * defaultFocalRatio = 1 (default)
 * defaultOffsetX = 0 (default)
 * defaultOffsetY = 0 (default)
 * errorOnMissingColorProfile = 1 (default)
 * groupCameraFallback =  Unknown Type "20EGroupCameraFallback"
 * imageFolder = "" (default)
 * input = "/tmp/tmpo_1no8yp/CameraInit/961e54591174ec5a2457c66da8eadc0cb03d89ba/viewpoints.sfm"
 * lensCorrectionProfileInfo = ""
 * lensCorrectionProfileSearchIgnoreCameraModel = 1
 * maxCoresAvailable =  Unknown Type "j" (default)
 * maxMemory

copy the output back to your g drive.

Then use something like meshlab to view them. or an online .obj viewer (remember it takes 3 files to make a 3d model)

In [11]:
!mkdir -p "/content/drive/MyDrive/meshroom_output"
!cp -r /content/output/* "/content/drive/MyDrive/meshroom_output/"

and then proceed from your google drive. Code below might work to allow you to display within this notebook.

In [12]:
%%capture
!pip install -q trimesh pillow gradio

In [15]:

import trimesh
import gradio as gr

# Pack OBJ + MTL + PNG textures into one .glb file
scene = trimesh.load('/content/output/texturedMesh.obj', process=False)
scene.export('/content/preview.glb')

with gr.Blocks() as demo:
    gr.Model3D(value='/content/preview.glb', height=600)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://a01533d6da75c1af45.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [14]:
## If your image above came out all gray-scaled colour, it might be that meshroom
## wrote the textur (colour information) as .exr format.

## this block of code will fix that by making a png, and will fix the .mtl file which tells the computer how to drape the colour on the 3dobject
## to use the png file.
## Once you run this file, run the block above again.
## then this will make the texture nicer
import os, glob
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"
import cv2
import numpy as np

out_dir = "/content/output"

for exr_path in sorted(glob.glob(f"{out_dir}/*.exr")):
    img = cv2.imread(exr_path, cv2.IMREAD_UNCHANGED)
    if img is None:
        raise RuntimeError(f"Could not read {exr_path}")
    # Drop alpha if present, clip to [0, 1], and apply linear -> sRGB gamma
    bgr = img[..., :3] if img.ndim == 3 else img
    srgb = np.power(np.clip(bgr, 0.0, 1.0), 1.0 / 2.2)
    png_path = os.path.splitext(exr_path)[0] + ".png"
    cv2.imwrite(png_path, (srgb * 255.0).astype(np.uint8))
    print("Saved:", png_path)

for mtl_path in glob.glob(f"{out_dir}/*.mtl"):
    with open(mtl_path, "r") as f:
        mtl = f.read()
    with open(mtl_path, "w") as f:
        f.write(mtl.replace(".exr", ".png"))

Saved: /content/output/texture_1001.png
